# Notebook 48 - MATLAB 2-state Kalman gate

This notebook starts the Python port of MATLAB UltraTimTrack's 2-state Kalman correction as a separate downstream gate.

The new code lives in `ultrasound_tracker.ultratimtrack_matlab_2state` and is intentionally separate from the experimental 4-state filter in `ultrasound_tracker.ultratimtrack_kalman`.

MATLAB state contract:

- State: `[superficial fascicle attachment x, fascicle alpha]`
- Process noise: `Q_parameter * dx^2`
- x measurement: fixed initial superficial fascicle x, with variance `X`
- alpha measurement: TimTrack/Hough alpha, with variance `R(1)`
- Backward smoother: MATLAB's RTS-style pass using `Pcorr / Ppred`
- Reconstruct final geometry from state plus current aponeuroses

This notebook tests the port with oracle and Python-input variants, then answers whether we can run an end-to-end pipeline yet.

In [1]:
from pathlib import Path
import json
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

import numpy as np
import pandas as pd
from scipy.io import loadmat

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultrasound_tracker.geometry import line_angles_batch, line_lengths_batch, normalize_angle
from ultrasound_tracker.matlab_compat import metric_row
from ultrasound_tracker.ultratimtrack_matlab_2state import (
    MatlabTwoStateKalmanConfig,
    run_matlab_2state_kalman,
)

OUT = ROOT / "results" / "notebook48_matlab_2state_kalman_gate"
OUT.mkdir(parents=True, exist_ok=True)

MATLAB_EXPORT = Path("/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat")
NB44_NPZ = ROOT / "results" / "notebook44_ultratrack_klt_oracle_mask_gate" / "opencv_ultratrack_klt_oracle_mask_features_arrays.npz"
NB45_NPZ = ROOT / "results" / "notebook45_ultratrack_klt_one_step_affine_diagnostics" / "opencv_ultratrack_klt_one_step_affine_arrays.npz"
PY_TIMTRACK_NPZ = ROOT / "results" / "notebook38_matlab_filter_usimage_mean71" / "Test2_matlab_filter_usimage_mean71_features_arrays.npz"

for path in [MATLAB_EXPORT, NB44_NPZ, NB45_NPZ, PY_TIMTRACK_NPZ]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Output folder:", OUT)

Output folder: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook48_matlab_2state_kalman_gate


## Load MATLAB reference and Python inputs

The primary reference is MATLAB final output from `UTT_numeric_export.mat`:

- `Region.fas_length`
- `Region.fas_pen`
- `Region.fas_ang`
- `Region.Fascicle.fas_x/fas_y`
- `Region.Fascicle.X_plus/X_minus/fas_p/fas_p_minus`

For the first gate, the aponeuroses are MATLAB-exported aponeuroses. That isolates the fascicle 2-state filter before we port the separate aponeurosis state estimator.

In [2]:
mat_root = loadmat(MATLAB_EXPORT, simplify_cells=True)["UTT_numeric_export"]
region = mat_root["Region"]
fascicle = region["Fascicle"]
geofeatures = list(np.asarray(mat_root["geofeatures"], dtype=object).reshape(-1))

height = int(mat_root["vidHeight"])
width = int(mat_root["vidWidth"])
mat_n = int(mat_root["NumFrames"])
mm_per_px = float(mat_root["ID"]) / height


def matlab_pair_series(values: object, n: int | None = None) -> np.ndarray:
    out = []
    for value in np.asarray(values, dtype=object).reshape(-1):
        arr = np.asarray(value, dtype=float).reshape(-1)
        out.append(arr[:2] if arr.size >= 2 else [np.nan, np.nan])
    result = np.asarray(out, dtype=float)
    return result if n is None else result[:n]


def matlab_fascicle_segments(x_field: str, y_field: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(fascicle[x_field], n=n)
    y = matlab_pair_series(fascicle[y_field], n=n)
    # MATLAB order is [deep; superficial]. Python convention is [sup, deep].
    return np.column_stack([x[:, 1], y[:, 1], x[:, 0], y[:, 0]])


def matlab_apo_segments(prefix: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(region[f"{prefix}_x"], n=n)
    y = matlab_pair_series(region[f"{prefix}_y"], n=n)
    return np.column_stack([x[:, 0], y[:, 0], x[:, 1], y[:, 1]])


def matlab_state_series(field: str, n: int | None = None) -> np.ndarray:
    return matlab_pair_series(fascicle[field], n=n)


def normalized_angles_deg(segments: np.ndarray) -> np.ndarray:
    return np.asarray([normalize_angle(v, degrees=True) for v in line_angles_batch(segments, degrees=True)], dtype=float)


mat_raw_klt = matlab_fascicle_segments("fas_x_original", "fas_y_original", n=mat_n)
mat_final_segments = matlab_fascicle_segments("fas_x", "fas_y", n=mat_n)
mat_sup = matlab_apo_segments("sup", n=mat_n)
mat_deep = matlab_apo_segments("deep", n=mat_n)
mat_timtrack_alpha = np.asarray([float(entry["alpha"]) for entry in geofeatures], dtype=float)

mat_final = {
    "FL_mm": np.asarray(region["fas_length"], dtype=float).reshape(-1)[:mat_n],
    "PEN_deg": np.asarray(region["fas_pen"], dtype=float).reshape(-1)[:mat_n],
    "ANG_deg": np.asarray(region["fas_ang"], dtype=float).reshape(-1)[:mat_n],
    "X_plus": matlab_state_series("X_plus", n=mat_n),
    "X_minus": matlab_state_series("X_minus", n=mat_n),
    "fas_p": matlab_state_series("fas_p", n=mat_n),
    "fas_p_minus": matlab_state_series("fas_p_minus", n=mat_n),
}

nb44 = np.load(NB44_NPZ, allow_pickle=True)
nb45 = np.load(NB45_NPZ, allow_pickle=True)
py_timtrack = np.load(PY_TIMTRACK_NPZ, allow_pickle=True)

py_seq_klt = np.asarray(nb44["fascicle_segments"], dtype=float)[:mat_n]
py_one_step_klt = np.asarray(nb45["fascicle_segments"], dtype=float)[:mat_n]
py_timtrack_alpha = np.asarray(py_timtrack["ANG_deg"], dtype=float).reshape(-1)[:mat_n]

config = MatlabTwoStateKalmanConfig(
    q_parameter=float(mat_root["Q"]),
    x_measurement_variance=float(mat_root["X"]),
    alpha_measurement_variance=float(np.asarray(mat_root["R"], dtype=float).reshape(-1)[0]),
    n_start_frames=int(mat_root["NS"]),
    run_smoother=True,
)

print({
    "frames": mat_n,
    "mm_per_px": mm_per_px,
    "config": config,
    "mat_raw_klt_shape": mat_raw_klt.shape,
    "py_seq_klt_shape": py_seq_klt.shape,
    "py_one_step_klt_shape": py_one_step_klt.shape,
})

{'frames': 2666, 'mm_per_px': 0.09021352313167261, 'config': MatlabTwoStateKalmanConfig(q_parameter=0.01, x_measurement_variance=100.0, alpha_measurement_variance=3.055292111054342, n_start_frames=1, run_smoother=True), 'mat_raw_klt_shape': (2666, 4), 'py_seq_klt_shape': (2666, 4), 'py_one_step_klt_shape': (2666, 4)}


## Run 2-state Kalman variants

Variants:

- `oracle_matlab_inputs`: MATLAB raw KLT + MATLAB TimTrack alpha + MATLAB aponeuroses. This isolates the Python 2-state port.
- `python_one_step_klt`: notebook 45 one-step Python KLT + MATLAB TimTrack alpha + MATLAB aponeuroses. This tests the best current Python KLT prior.
- `python_sequential_klt`: notebook 44 sequential Python KLT + MATLAB TimTrack alpha + MATLAB aponeuroses. This tests the current sequential KLT prior.
- `python_sequential_klt_python_timtrack`: sequential KLT + current Python TimTrack alpha. This is the closest current end-to-end scaffold, still using MATLAB aponeuroses because the aponeurosis Kalman estimator has not been ported yet.

In [3]:
variants = {
    "oracle_matlab_inputs": {
        "klt_segments": mat_raw_klt,
        "alpha": mat_timtrack_alpha,
        "description": "MATLAB raw KLT + MATLAB TimTrack alpha + MATLAB aponeuroses",
    },
    "python_one_step_klt": {
        "klt_segments": py_one_step_klt,
        "alpha": mat_timtrack_alpha,
        "description": "Notebook45 one-step Python KLT + MATLAB TimTrack alpha + MATLAB aponeuroses",
    },
    "python_sequential_klt": {
        "klt_segments": py_seq_klt,
        "alpha": mat_timtrack_alpha,
        "description": "Notebook44 sequential Python KLT + MATLAB TimTrack alpha + MATLAB aponeuroses",
    },
    "python_sequential_klt_python_timtrack": {
        "klt_segments": py_seq_klt,
        "alpha": py_timtrack_alpha,
        "description": "Notebook44 sequential Python KLT + Python TimTrack alpha + MATLAB aponeuroses",
    },
}

results = {}
for name, spec in variants.items():
    results[name] = run_matlab_2state_kalman(
        spec["klt_segments"],
        spec["alpha"],
        mat_sup,
        mat_deep,
        config=config,
        mm_per_pixel=mm_per_px,
    )

out_npz = OUT / "matlab_2state_kalman_variant_arrays.npz"
np.savez(
    out_npz,
    frame=np.arange(mat_n, dtype=np.int32),
    matlab_FL_mm=mat_final["FL_mm"],
    matlab_PEN_deg=mat_final["PEN_deg"],
    matlab_ANG_deg=mat_final["ANG_deg"],
    matlab_X_plus=mat_final["X_plus"],
    matlab_X_minus=mat_final["X_minus"],
    oracle_FL_mm=results["oracle_matlab_inputs"]["FL_mm"],
    oracle_PEN_deg=results["oracle_matlab_inputs"]["PEN_deg"],
    oracle_ANG_deg=results["oracle_matlab_inputs"]["ANG_deg"],
    python_one_step_FL_mm=results["python_one_step_klt"]["FL_mm"],
    python_one_step_PEN_deg=results["python_one_step_klt"]["PEN_deg"],
    python_one_step_ANG_deg=results["python_one_step_klt"]["ANG_deg"],
    python_seq_FL_mm=results["python_sequential_klt"]["FL_mm"],
    python_seq_PEN_deg=results["python_sequential_klt"]["PEN_deg"],
    python_seq_ANG_deg=results["python_sequential_klt"]["ANG_deg"],
    python_seq_python_timtrack_FL_mm=results["python_sequential_klt_python_timtrack"]["FL_mm"],
    python_seq_python_timtrack_PEN_deg=results["python_sequential_klt_python_timtrack"]["PEN_deg"],
    python_seq_python_timtrack_ANG_deg=results["python_sequential_klt_python_timtrack"]["ANG_deg"],
)
print("Saved:", out_npz)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook48_matlab_2state_kalman_gate/matlab_2state_kalman_variant_arrays.npz


## Gate metrics against MATLAB final output

Targets for final output gate:

- `FL` RMSE < 2 mm
- `PEN` RMSE < 1 degree
- `ANG` RMSE < 1 degree

In [4]:
def metric(label: str, reference, estimate, unit: str, target_rmse: float | None = None) -> dict:
    row = metric_row(label, reference, estimate)
    row["unit"] = unit
    row["target_rmse"] = target_rmse
    row["passes"] = bool(target_rmse is None or row["rmse"] <= target_rmse)
    return row


metric_rows = []
for name, result in results.items():
    rows = [
        metric(f"{name}_FL_mm", mat_final["FL_mm"], result["FL_mm"], "mm", target_rmse=2.0),
        metric(f"{name}_PEN_deg", mat_final["PEN_deg"], result["PEN_deg"], "deg", target_rmse=1.0),
        metric(f"{name}_ANG_deg", mat_final["ANG_deg"], result["ANG_deg"], "deg", target_rmse=1.0),
        metric(f"{name}_state_x_sup", mat_final["X_plus"][:, 0], result["X_plus"][:, 0], "px", target_rmse=2.0),
        metric(f"{name}_state_alpha", mat_final["X_plus"][:, 1], result["X_plus"][:, 1], "deg", target_rmse=1.0),
    ]
    for row in rows:
        row["variant"] = name
        row["description"] = variants[name]["description"]
    metric_rows.extend(rows)

metrics = pd.DataFrame(metric_rows)
metrics = metrics[["variant", "description", "comparison", "unit", "n", "bias", "mae", "rmse", "corr", "target_rmse", "passes"]]
metrics_path = OUT / "matlab_2state_kalman_gate_metrics.csv"
metrics.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)
display(metrics)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook48_matlab_2state_kalman_gate/matlab_2state_kalman_gate_metrics.csv


,variant,description,comparison,unit,n,bias,mae,rmse,corr,target_rmse,passes
0,oracle_matlab_inputs,MATLAB raw KLT + MATLAB TimTrack alpha + MATLA...,oracle_matlab_inputs_FL_mm,mm,2666,0.006937,0.405313,0.560316,0.998236,2.0,True
1,oracle_matlab_inputs,MATLAB raw KLT + MATLAB TimTrack alpha + MATLA...,oracle_matlab_inputs_PEN_deg,deg,2666,0.003438,0.203201,0.265666,0.998158,1.0,True
2,oracle_matlab_inputs,MATLAB raw KLT + MATLAB TimTrack alpha + MATLA...,oracle_matlab_inputs_ANG_deg,deg,2666,0.003438,0.203201,0.265666,0.998355,1.0,True
3,oracle_matlab_inputs,MATLAB raw KLT + MATLAB TimTrack alpha + MATLA...,oracle_matlab_inputs_state_x_sup,px,2666,1.355793,3.255640,4.034723,0.932617,2.0,False
4,oracle_matlab_inputs,MATLAB raw KLT + MATLAB TimTrack alpha + MATLA...,oracle_matlab_inputs_state_alpha,deg,2666,-0.054285,0.284443,0.351644,0.997168,1.0,True
5,python_one_step_klt,Notebook45 one-step Python KLT + MATLAB TimTra...,python_one_step_klt_FL_mm,mm,2666,-0.024759,0.566556,0.752955,0.996827,2.0,True
6,python_one_step_klt,Notebook45 one-step Python KLT + MATLAB TimTra...,python_one_step_klt_PEN_deg,deg,2666,0.025406,0.297381,0.409724,0.995667,1.0,True
7,python_one_step_klt,Notebook45 one-step Python KLT + MATLAB TimTra...,python_one_step_klt_ANG_deg,deg,2666,0.025406,0.297381,0.409724,0.996128,1.0,True
8,python_one_step_klt,Notebook45 one-step Python KLT + MATLAB TimTra...,python_one_step_klt_state_x_sup,px,2666,1.493668,3.391745,4.182722,0.919625,2.0,False
9,python_one_step_klt,Notebook45 one-step Python KLT + MATLAB TimTra...,python_one_step_klt_state_alpha,deg,2666,-0.032317,0.370551,0.496451,0.994285,1.0,True


In [5]:
summary_rows = []
for name in variants:
    group = metrics[metrics["variant"] == name]
    final_group = group[group["comparison"].str.endswith(("FL_mm", "PEN_deg", "ANG_deg"))]
    summary_rows.append({
        "variant": name,
        "description": variants[name]["description"],
        "FL_rmse_mm": float(group.loc[group["comparison"] == f"{name}_FL_mm", "rmse"].iloc[0]),
        "PEN_rmse_deg": float(group.loc[group["comparison"] == f"{name}_PEN_deg", "rmse"].iloc[0]),
        "ANG_rmse_deg": float(group.loc[group["comparison"] == f"{name}_ANG_deg", "rmse"].iloc[0]),
        "state_x_rmse_px": float(group.loc[group["comparison"] == f"{name}_state_x_sup", "rmse"].iloc[0]),
        "state_alpha_rmse_deg": float(group.loc[group["comparison"] == f"{name}_state_alpha", "rmse"].iloc[0]),
        "passes_final_gate": bool(final_group["passes"].all()),
        "passes_state_gate": bool(group[group["comparison"].str.contains("state")]["passes"].all()),
    })
summary_df = pd.DataFrame(summary_rows)
summary_table_path = OUT / "matlab_2state_kalman_variant_summary.csv"
summary_df.to_csv(summary_table_path, index=False)
print("Saved:", summary_table_path)
display(summary_df)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook48_matlab_2state_kalman_gate/matlab_2state_kalman_variant_summary.csv


,variant,description,FL_rmse_mm,PEN_rmse_deg,ANG_rmse_deg,state_x_rmse_px,state_alpha_rmse_deg,passes_final_gate,passes_state_gate
0,oracle_matlab_inputs,MATLAB raw KLT + MATLAB TimTrack alpha + MATLA...,0.560316,0.265666,0.265666,4.034723,0.351644,True,False
1,python_one_step_klt,Notebook45 one-step Python KLT + MATLAB TimTra...,0.752955,0.409724,0.409724,4.182722,0.496451,True,False
2,python_sequential_klt,Notebook44 sequential Python KLT + MATLAB TimT...,6.868820,3.292758,3.292758,9.262139,3.272790,False,False
3,python_sequential_klt_python_timtrack,Notebook44 sequential Python KLT + Python TimT...,11.208491,5.429017,5.429017,9.262139,5.479536,False,False


## Nine-frame final-output check

This uses the same focus frames as the earlier TimTrack notebooks. It is a small validation slice, not a replacement for the full-sequence gate above.

In [6]:
focus_frames = np.asarray([0, 122, 533, 691, 955, 1066, 1600, 2133, 2665], dtype=int)
focus_rows = []
for name, result in results.items():
    for comparison, ref_key, est_key, unit, target in [
        ("FL_mm", "FL_mm", "FL_mm", "mm", 2.0),
        ("PEN_deg", "PEN_deg", "PEN_deg", "deg", 1.0),
        ("ANG_deg", "ANG_deg", "ANG_deg", "deg", 1.0),
    ]:
        row = metric(
            f"{name}_{comparison}",
            mat_final[ref_key][focus_frames],
            result[est_key][focus_frames],
            unit,
            target_rmse=target,
        )
        row["variant"] = name
        focus_rows.append(row)
focus_metrics = pd.DataFrame(focus_rows)
focus_metrics = focus_metrics[["variant", "comparison", "unit", "n", "bias", "mae", "rmse", "corr", "target_rmse", "passes"]]
focus_path = OUT / "matlab_2state_kalman_focus9_metrics.csv"
focus_metrics.to_csv(focus_path, index=False)
print("Saved:", focus_path)
display(focus_metrics)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook48_matlab_2state_kalman_gate/matlab_2state_kalman_focus9_metrics.csv


,variant,comparison,unit,n,bias,mae,rmse,corr,target_rmse,passes
0,oracle_matlab_inputs,oracle_matlab_inputs_FL_mm,mm,9,0.156696,0.591220,0.781838,0.997593,2.0,True
1,oracle_matlab_inputs,oracle_matlab_inputs_PEN_deg,deg,9,-0.100735,0.256438,0.321480,0.998160,1.0,True
2,oracle_matlab_inputs,oracle_matlab_inputs_ANG_deg,deg,9,-0.100735,0.256438,0.321480,0.998456,1.0,True
3,python_one_step_klt,python_one_step_klt_FL_mm,mm,9,0.500311,0.929414,1.224210,0.995918,2.0,True
4,python_one_step_klt,python_one_step_klt_PEN_deg,deg,9,-0.352915,0.501146,0.745082,0.994781,1.0,True
5,python_one_step_klt,python_one_step_klt_ANG_deg,deg,9,-0.352915,0.501146,0.745082,0.995627,1.0,True
6,python_sequential_klt,python_sequential_klt_FL_mm,mm,9,-4.825937,6.060312,7.409284,0.875262,2.0,False
7,python_sequential_klt,python_sequential_klt_PEN_deg,deg,9,1.781900,2.672835,3.106874,0.864522,1.0,False
8,python_sequential_klt,python_sequential_klt_ANG_deg,deg,9,1.781900,2.672835,3.106874,0.896856,1.0,False
9,python_sequential_klt_python_timtrack,python_sequential_klt_python_timtrack_FL_mm,mm,9,3.367805,3.367805,4.792993,0.994972,2.0,False


## End-to-end readiness

The important distinction:

- The **2-state Kalman code port is ready as a downstream gate** when fed a good KLT prior and MATLAB-style alpha measurements.
- The current **full Python end-to-end scaffold is not ready** because sequential KLT raw parity and full Python TimTrack alpha still fail the downstream final gate.
- The aponeurosis Kalman estimator is not ported here; this notebook uses MATLAB aponeuroses to isolate the fascicle 2-state model.

In [7]:
oracle_pass = bool(summary_df.loc[summary_df["variant"] == "oracle_matlab_inputs", "passes_final_gate"].iloc[0])
py_one_step_pass = bool(summary_df.loc[summary_df["variant"] == "python_one_step_klt", "passes_final_gate"].iloc[0])
py_seq_pass = bool(summary_df.loc[summary_df["variant"] == "python_sequential_klt", "passes_final_gate"].iloc[0])
py_seq_pyalpha_pass = bool(summary_df.loc[summary_df["variant"] == "python_sequential_klt_python_timtrack", "passes_final_gate"].iloc[0])

readiness = pd.DataFrame([
    {
        "gate": "TimTrack mask/doHough",
        "status": "passed on corrected 9-frame gate; full Python alpha still needs downstream validation",
        "ready_for_next": True,
    },
    {
        "gate": "KLT local one-step prior",
        "status": "good enough for 2-state Kalman gate" if py_one_step_pass else "not enough for 2-state Kalman gate",
        "ready_for_next": py_one_step_pass,
    },
    {
        "gate": "KLT sequential raw prior",
        "status": "not passed; current sequential prior fails 2-state final gate",
        "ready_for_next": False,
    },
    {
        "gate": "MATLAB fascicle 2-state Kalman",
        "status": "ported and passes oracle final-output gate" if oracle_pass else "ported but oracle gate failed",
        "ready_for_next": oracle_pass,
    },
    {
        "gate": "Current Python end-to-end scaffold",
        "status": "passes" if py_seq_pyalpha_pass else "not ready; KLT sequential and full Python alpha still fail final gate",
        "ready_for_next": py_seq_pyalpha_pass,
    },
])
readiness_path = OUT / "readiness.csv"
readiness.to_csv(readiness_path, index=False)
print("Saved:", readiness_path)
display(readiness)

summary = {
    "module": "ultrasound_tracker.ultratimtrack_matlab_2state",
    "oracle_final_gate_passes": oracle_pass,
    "python_one_step_klt_final_gate_passes": py_one_step_pass,
    "python_sequential_klt_final_gate_passes": py_seq_pass,
    "python_sequential_klt_python_timtrack_final_gate_passes": py_seq_pyalpha_pass,
    "oracle_FL_rmse_mm": float(summary_df.loc[summary_df["variant"] == "oracle_matlab_inputs", "FL_rmse_mm"].iloc[0]),
    "oracle_PEN_rmse_deg": float(summary_df.loc[summary_df["variant"] == "oracle_matlab_inputs", "PEN_rmse_deg"].iloc[0]),
    "oracle_ANG_rmse_deg": float(summary_df.loc[summary_df["variant"] == "oracle_matlab_inputs", "ANG_rmse_deg"].iloc[0]),
    "python_one_step_FL_rmse_mm": float(summary_df.loc[summary_df["variant"] == "python_one_step_klt", "FL_rmse_mm"].iloc[0]),
    "python_sequential_FL_rmse_mm": float(summary_df.loc[summary_df["variant"] == "python_sequential_klt", "FL_rmse_mm"].iloc[0]),
    "current_end_to_end_FL_rmse_mm": float(summary_df.loc[summary_df["variant"] == "python_sequential_klt_python_timtrack", "FL_rmse_mm"].iloc[0]),
    "answer_can_run_end_to_end": "A scaffold can run, but the complete Python pipeline is not ready for the final gate until sequential KLT and full-sequence TimTrack alpha are corrected.",
    "next_action": "Port aponeurosis state estimator or fix sequential KLT prior; keep the experimental 4-state filter separate.",
}
summary_path = OUT / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved:", summary_path)
print(json.dumps(summary, indent=2))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook48_matlab_2state_kalman_gate/readiness.csv


,gate,status,ready_for_next
0,TimTrack mask/doHough,passed on corrected 9-frame gate; full Python ...,True
1,KLT local one-step prior,good enough for 2-state Kalman gate,True
2,KLT sequential raw prior,not passed; current sequential prior fails 2-s...,False
3,MATLAB fascicle 2-state Kalman,ported and passes oracle final-output gate,True
4,Current Python end-to-end scaffold,not ready; KLT sequential and full Python alph...,False


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook48_matlab_2state_kalman_gate/summary.json
{
  "module": "ultrasound_tracker.ultratimtrack_matlab_2state",
  "oracle_final_gate_passes": true,
  "python_one_step_klt_final_gate_passes": true,
  "python_sequential_klt_final_gate_passes": false,
  "python_sequential_klt_python_timtrack_final_gate_passes": false,
  "oracle_FL_rmse_mm": 0.5603162940100396,
  "oracle_PEN_rmse_deg": 0.26566580622257935,
  "oracle_ANG_rmse_deg": 0.26566580622257935,
  "python_one_step_FL_rmse_mm": 0.7529551324064312,
  "python_sequential_FL_rmse_mm": 6.868819502554928,
  "current_end_to_end_FL_rmse_mm": 11.208491029219624,
  "answer_can_run_end_to_end": "A scaffold can run, but the complete Python pipeline is not ready for the final gate until sequential KLT and full-sequence TimTrack alpha are corrected.",
  "next_action": "Port aponeurosis state estimator or fix sequential KLT prior; keep the experimental 4-state filter separate."
}
